### 앙상블 
- 단일 결정트리(DecisionTree)의 단점을 극복하기 위해 여러 머신러닝 모델을 연결하여 더 강력한 모델을 만드는 과정 
- 주어진 자료로부터 여러개의 예측 모형을 만든 후 예측 모형을 조합하여 하나의 예측 모형을 생성하는 과정 
- 대표적인 기법 : 배깅, 부스팅, 랜덤포레스트


#### 배깅 
- 주어진 자료를 모집단으로 간주하여 주어진 자료에서 여러 개의 부트스트랩 자료를 생성하고 각각의 부트스트랩 자료에 예측 모형을 만든 후 결합하여 최종 모델 완성 
- 장점 
    - 분산을 줄이고 과적합의 문제를 완화
    - 안정적인 성능 보장 
    - 비선형 구조의 데이터와 노이즈 데이터에 대한 문제가 완화
- parameter 
    - estimator
        - 기본값 : None
        - 기본 모델을 설정
    - n_estimator
        - 기본값 : 10
        - 생성시킬 모델의 개수
    - max_samples
        - 기본값 : 1.0
        - 생성이 된 모델들이 사용할 데이터의 개수(인덱스의 수) 비율
    - max_features
        - 기본값 : 1.0
        - 생성이 된 모델들이 사용할 피쳐(컬럼)의 수 비율
    - oob_score
        - 기본값 : False
        - oob(Out-Of-Bag) 데이터로 일반화 성능을 평가할것인가?
    - bootstrap
        - 기본값 : True
        - 샘플링 데이터를 이용 시 중복 데이터를 허용
        - False인 경우에는 각각의 모델들에 max_samples의 비율이 1.0이라면 모두 같은 데이터를 학습으로 사용( 다양성이 부족 )
    - bootstrap_features
        - 기본값 : False
        - 샘플링 데이터 이용시 중복 컬럼을 허용할것인가
    - n_jobs
        - 기본값 : None
        - CPU의 병렬처리 개수( -1을 사용시에는 모든 CPU를 사용 )
- 속성 
    - estimators_
        - 학습된 모델의 리스트
    - estimators_samples_
        - 각 모델이 학습한 샘플의 인덱스(행의 위치)
    - estimators_features_
        - 각 모델이 학습한 컬럼의 인덱스(열의 위치)
    - oob_score_
        - oob 데이터를 기반으로 정확도(분류) / R2(회귀)
    - oob_decision_function_
        - OOB 데이터의 클래스별 예측 확률
- 메서드 
    - fit_predict()
        - 학습 후 예측 수행 (학습 데이터로 예측)

In [1]:
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import f1_score

In [2]:
hotel = pd.read_csv("../data/hotel_bookings.csv")

In [3]:
hotel.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 11 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   is_canceled                     20000 non-null  int64  
 1   deposit_type                    20000 non-null  object 
 2   lead_time                       19995 non-null  float64
 3   stays_in_weekend_nights         20000 non-null  int64  
 4   stays_in_week_nights            20000 non-null  int64  
 5   is_repeated_guest               19642 non-null  float64
 6   previous_cancellations          20000 non-null  int64  
 7   previous_bookings_not_canceled  20000 non-null  int64  
 8   booking_changes                 20000 non-null  int64  
 9   days_in_waiting_list            20000 non-null  int64  
 10  adr                             18937 non-null  float64
dtypes: float64(3), int64(7), object(1)
memory usage: 1.7+ MB


- is_canceled : 예약의 취소 여부 ( 1 = 취소, 0 = 체크인 ) (target 데이터)
- deposit_type : 보증금의 유형 (문자열) --> 더미변수 생성
- lead_time : 예약 시차 ( 예약일 ~ 체크인일 )
- stays_in_weekend_nights : 주말 숙박 일수
- stays_in_week_nights : 평일 숙박 일수
- is_repeated_guest : 재방문 고객 여부 (1 = 재방문, 0 = 신규)
- previous_cancellations : 과거 취소 횟수
- previous_bookings_not_canceled : 과거 정상 투숙 횟수
- booking_changes : 예약 변경 횟수
- days_in_waiting_list : 대기 명단에 있었던 횟수
- adr : 1박당 평균 객실의 요금

In [5]:
hotel['lead_time'].describe()

count    19995.000000
mean        85.978345
std         96.427240
min          0.000000
25%         11.000000
50%         51.000000
75%        132.000000
max        629.000000
Name: lead_time, dtype: float64

In [6]:
hotel['is_repeated_guest'].describe()

count    19642.000000
mean         0.038133
std          0.191521
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: is_repeated_guest, dtype: float64

In [7]:
hotel['is_repeated_guest'].value_counts()

is_repeated_guest
0.0    18893
1.0      749
Name: count, dtype: int64

In [8]:
hotel['adr'].describe()

count    18937.000000
mean       101.410239
std         49.245097
min         -6.380000
25%         68.800000
50%         94.500000
75%        126.000000
max        451.500000
Name: adr, dtype: float64

In [10]:
# adr 데이터에서 음수인 데이터를 확인 
flag = hotel['adr'] < 0
hotel = hotel.loc[~flag, ]

In [11]:
hotel['adr'].describe()

count    18936.000000
mean       101.415931
std         49.240166
min          0.000000
25%         68.822500
50%         94.500000
75%        126.000000
max        451.500000
Name: adr, dtype: float64

In [12]:
hotel.loc[ hotel['adr'] == 0,  ]

,is_canceled,deposit_type,lead_time,stays_in_weekend_nights,stays_in_week_nights,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,days_in_waiting_list,adr
42,0,No Deposit,97.0,0,2,0.0,0,0,0,0,0.0
46,0,No Deposit,0.0,0,1,1.0,0,3,0,0,0.0
80,0,No Deposit,8.0,0,0,0.0,0,0,0,0,0.0
108,0,No Deposit,30.0,2,3,0.0,0,0,4,14,0.0
119,0,No Deposit,4.0,0,2,0.0,0,0,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
18558,1,No Deposit,244.0,2,4,0.0,0,0,0,0,0.0
19141,1,No Deposit,71.0,1,1,0.0,0,0,0,0,0.0
19223,1,No Deposit,1.0,0,1,1.0,4,11,0,0,0.0
19253,1,No Deposit,23.0,1,1,0.0,0,0,1,0,0.0


In [13]:
# lead_time, adr 컬럼의 결측치들은 평균값 | 중간값으로 대체 
hotel['lead_time'] = hotel['lead_time'].fillna( hotel['lead_time'].mean() )
hotel['adr'] = hotel['adr'].fillna(hotel['adr'].mean())

In [14]:
# 결측치 개수 확인 isna() + sum()
hotel.isna().sum()

is_canceled                         0
deposit_type                        0
lead_time                           0
stays_in_weekend_nights             0
stays_in_week_nights                0
is_repeated_guest                 358
previous_cancellations              0
previous_bookings_not_canceled      0
booking_changes                     0
days_in_waiting_list                0
adr                                 0
dtype: int64

In [17]:
hotel['is_repeated_guest'].value_counts().index[0]

np.float64(0.0)

In [18]:
# is_repeated_guest 컬럼의 결측치는 해당 컬럼의 최빈값으로 채워준다. ( 범주형 데이터이기때문에 )
hotel['is_repeated_guest'] = hotel['is_repeated_guest'].fillna(
    hotel['is_repeated_guest'].value_counts().index[0]
)

In [19]:
# deposit_type 컬럼은 문자형 데이터 -> 문자의 유형들을 확인 
hotel['deposit_type'].unique()

array(['No Deposit', 'Refundable', 'Non Refund'], dtype=object)

In [20]:
hotel['deposit_type'].value_counts()

deposit_type
No Deposit    19137
Non Refund      834
Refundable       28
Name: count, dtype: int64

In [21]:
df = pd.get_dummies(hotel, columns=['deposit_type'], drop_first=True)

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19999 entries, 0 to 19999
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   is_canceled                     19999 non-null  int64  
 1   lead_time                       19999 non-null  float64
 2   stays_in_weekend_nights         19999 non-null  int64  
 3   stays_in_week_nights            19999 non-null  int64  
 4   is_repeated_guest               19999 non-null  float64
 5   previous_cancellations          19999 non-null  int64  
 6   previous_bookings_not_canceled  19999 non-null  int64  
 7   booking_changes                 19999 non-null  int64  
 8   days_in_waiting_list            19999 non-null  int64  
 9   adr                             19999 non-null  float64
 10  deposit_type_Non Refund         19999 non-null  bool   
 11  deposit_type_Refundable         19999 non-null  bool   
dtypes: bool(2), float64(3), int64(7)
memo

In [23]:
df['is_canceled'].value_counts()

is_canceled
0    17599
1     2400
Name: count, dtype: int64

In [24]:
# 데이터의 불균형을 유지한채 독립 , 종속 변수 
x = df.drop('is_canceled', axis=1)
y = df['is_canceled']

In [25]:
# train, test 데이터셋으로 나눠준다 
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=42
)

In [26]:
y_train.value_counts()

is_canceled
0    14079
1     1920
Name: count, dtype: int64

In [43]:
# 배깅 모델 생성 -> DecisionTree를 기본 모델로 사용 --> 데이터의 불균형이 심하다 --> class_weight
base_model = DecisionTreeClassifier(class_weight='balanced', max_depth=3)

model = BaggingClassifier(base_model, n_estimators=500)

In [44]:
model.fit(X_train, y_train)

,"estimator estimator: object, default=NoneThe base estimator to fit on random subsets of the dataset.If None, then the base estimator is a:class:`~sklearn.tree.DecisionTreeClassifier`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.","DecisionTreeC..., max_depth=3)"
,"n_estimators n_estimators: int, default=10The number of base estimators in the ensemble.",500
,"max_samples max_samples: int or float, default=NoneThe number of samples to draw from X to train each base estimator (withreplacement by default, see `bootstrap` for more details).- If None, then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples.",None
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator (without replacement by default, see `bootstrap_features` for moredetails).- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.",1.0
,"bootstrap bootstrap: bool, default=TrueWhether samples are drawn with replacement. If False, sampling withoutreplacement is performed. If fitting with `sample_weight`, it isstrongly recommended to choose True, as only drawing with replacementwill ensure the expected frequency semantics of `sample_weight`.",True
,"bootstrap_features bootstrap_features: bool, default=FalseWhether features are drawn with replacement.",False
,"oob_score oob_score: bool, default=FalseWhether to use out-of-bag samples to estimatethe generalization error. Only available if bootstrap=True.",False
,"warm_start warm_start: bool, default=FalseWhen set to True, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fita whole new ensemble. See :term:`the Glossary `... versionadded:: 0.17 *warm_start* constructor parameter.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for both :meth:`fit` and:meth:`predict`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random resampling of the original dataset(sample wise and feature wise).If the base estimator accepts a `random_state` attribute, a differentseed is generated for each instance in the ensemble.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",0


In [45]:
pred = model.predict(X_test)

In [46]:
print(
    round( f1_score(y_test, pred), 4 )
)

0.5748


- 데이터의 불균형을 샘플링 기법을 통해서 리샘플을 하고 모델에 학습 시켜서 성능을 확인 
- 랜덤오버샘플링 -> 2:1비율로 리샘플링
- 배깅 모델에서 부트스트랩에서 샘플의 개수를 0.8로 지정하고 모델의 개수는 100
- 의사결정나무에서 class_weight 기본값으로 최대 깊이는 3 사용하여 성능을 평가 


In [47]:
from imblearn.over_sampling import RandomOverSampler, SMOTE

In [48]:
oversample = RandomOverSampler(sampling_strategy=0.5)

In [49]:
x_over, y_over = oversample.fit_resample(x, y)

In [50]:
y_over.value_counts()

is_canceled
0    17599
1     8799
Name: count, dtype: int64

In [51]:
X_train, X_test, y_train, y_test = train_test_split(
    x_over, y_over, test_size=0.2, random_state=42, stratify=y_over
)

In [68]:
base_model = DecisionTreeClassifier()
clf = BaggingClassifier(estimator=base_model, n_estimators=100, max_samples=0.8)

In [69]:
clf.fit(X_train, y_train)

,"estimator estimator: object, default=NoneThe base estimator to fit on random subsets of the dataset.If None, then the base estimator is a:class:`~sklearn.tree.DecisionTreeClassifier`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",DecisionTreeClassifier()
,"n_estimators n_estimators: int, default=10The number of base estimators in the ensemble.",100
,"max_samples max_samples: int or float, default=NoneThe number of samples to draw from X to train each base estimator (withreplacement by default, see `bootstrap` for more details).- If None, then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples.",0.8
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator (without replacement by default, see `bootstrap_features` for moredetails).- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.",1.0
,"bootstrap bootstrap: bool, default=TrueWhether samples are drawn with replacement. If False, sampling withoutreplacement is performed. If fitting with `sample_weight`, it isstrongly recommended to choose True, as only drawing with replacementwill ensure the expected frequency semantics of `sample_weight`.",True
,"bootstrap_features bootstrap_features: bool, default=FalseWhether features are drawn with replacement.",False
,"oob_score oob_score: bool, default=FalseWhether to use out-of-bag samples to estimatethe generalization error. Only available if bootstrap=True.",False
,"warm_start warm_start: bool, default=FalseWhen set to True, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fita whole new ensemble. See :term:`the Glossary `... versionadded:: 0.17 *warm_start* constructor parameter.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for both :meth:`fit` and:meth:`predict`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random resampling of the original dataset(sample wise and feature wise).If the base estimator accepts a `random_state` attribute, a differentseed is generated for each instance in the ensemble.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",0


In [70]:
pred = clf.predict(X_test)

In [71]:
print(
    round( f1_score(y_test, pred), 4 )
)

0.9308


In [81]:
# SMOTE 방식으로 오버 샘플링 데이터는 1:1 비율로 생성
smote = SMOTE(sampling_strategy=0.5)

In [82]:
x_sm, y_sm = smote.fit_resample(x, y)

In [83]:
y_sm.value_counts()

is_canceled
0    17599
1     8799
Name: count, dtype: int64

In [84]:
clf2 = BaggingClassifier(estimator=base_model, n_estimators=100)

In [85]:
X_train, X_test, y_train, y_test = train_test_split(
    x_sm, y_sm, test_size=0.2, random_state=42, stratify=y_sm
)

In [86]:
clf2.fit(X_train, y_train)

,"estimator estimator: object, default=NoneThe base estimator to fit on random subsets of the dataset.If None, then the base estimator is a:class:`~sklearn.tree.DecisionTreeClassifier`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",DecisionTreeClassifier()
,"n_estimators n_estimators: int, default=10The number of base estimators in the ensemble.",100
,"max_samples max_samples: int or float, default=NoneThe number of samples to draw from X to train each base estimator (withreplacement by default, see `bootstrap` for more details).- If None, then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples.",None
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator (without replacement by default, see `bootstrap_features` for moredetails).- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.",1.0
,"bootstrap bootstrap: bool, default=TrueWhether samples are drawn with replacement. If False, sampling withoutreplacement is performed. If fitting with `sample_weight`, it isstrongly recommended to choose True, as only drawing with replacementwill ensure the expected frequency semantics of `sample_weight`.",True
,"bootstrap_features bootstrap_features: bool, default=FalseWhether features are drawn with replacement.",False
,"oob_score oob_score: bool, default=FalseWhether to use out-of-bag samples to estimatethe generalization error. Only available if bootstrap=True.",False
,"warm_start warm_start: bool, default=FalseWhen set to True, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fita whole new ensemble. See :term:`the Glossary `... versionadded:: 0.17 *warm_start* constructor parameter.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for both :meth:`fit` and:meth:`predict`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random resampling of the original dataset(sample wise and feature wise).If the base estimator accepts a `random_state` attribute, a differentseed is generated for each instance in the ensemble.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",0


In [87]:
pred2 = clf2.predict(X_test)

In [88]:
print(
    round( f1_score(y_test, pred2), 4 )
)

0.8285


In [89]:
# 기본 의사결정나무 학습
base_model.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the curre

In [90]:
pred3 = base_model.predict(X_test)

In [91]:
print(
    round( f1_score(y_test, pred3), 4 )
)

0.7773


In [98]:
# oob -> 부트스트랩에서 선택되지 않은 데이터 
clf_oob = BaggingClassifier(estimator=base_model, n_estimators=100, oob_score=True)

In [99]:
clf_oob.fit(x_over, y_over)

,"estimator estimator: object, default=NoneThe base estimator to fit on random subsets of the dataset.If None, then the base estimator is a:class:`~sklearn.tree.DecisionTreeClassifier`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",DecisionTreeClassifier()
,"n_estimators n_estimators: int, default=10The number of base estimators in the ensemble.",100
,"max_samples max_samples: int or float, default=NoneThe number of samples to draw from X to train each base estimator (withreplacement by default, see `bootstrap` for more details).- If None, then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples.",None
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator (without replacement by default, see `bootstrap_features` for moredetails).- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.",1.0
,"bootstrap bootstrap: bool, default=TrueWhether samples are drawn with replacement. If False, sampling withoutreplacement is performed. If fitting with `sample_weight`, it isstrongly recommended to choose True, as only drawing with replacementwill ensure the expected frequency semantics of `sample_weight`.",True
,"bootstrap_features bootstrap_features: bool, default=FalseWhether features are drawn with replacement.",False
,"oob_score oob_score: bool, default=FalseWhether to use out-of-bag samples to estimatethe generalization error. Only available if bootstrap=True.",True
,"warm_start warm_start: bool, default=FalseWhen set to True, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fita whole new ensemble. See :term:`the Glossary `... versionadded:: 0.17 *warm_start* constructor parameter.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for both :meth:`fit` and:meth:`predict`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random resampling of the original dataset(sample wise and feature wise).If the base estimator accepts a `random_state` attribute, a differentseed is generated for each instance in the ensemble.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",0


In [100]:
clf_oob.oob_score_

0.9619289340101523

In [101]:
# 배깅 회귀 -> 배깅용 회귀 모델을 로드 -> 기본 모델은 의사결정나무(회귀)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor

In [102]:
car = pd.read_csv("../data/CarPrice_Assignment.csv")

In [103]:
car.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 26 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   car_ID            205 non-null    int64  
 1   symboling         205 non-null    int64  
 2   CarName           205 non-null    object 
 3   fueltype          205 non-null    object 
 4   aspiration        205 non-null    object 
 5   doornumber        205 non-null    object 
 6   carbody           205 non-null    object 
 7   drivewheel        205 non-null    object 
 8   enginelocation    205 non-null    object 
 9   wheelbase         205 non-null    float64
 10  carlength         205 non-null    float64
 11  carwidth          205 non-null    float64
 12  carheight         205 non-null    float64
 13  curbweight        205 non-null    int64  
 14  enginetype        205 non-null    object 
 15  cylindernumber    205 non-null    object 
 16  enginesize        205 non-null    int64  
 1

In [104]:
car.columns

Index(['car_ID', 'symboling', 'CarName', 'fueltype', 'aspiration',
       'doornumber', 'carbody', 'drivewheel', 'enginelocation', 'wheelbase',
       'carlength', 'carwidth', 'carheight', 'curbweight', 'enginetype',
       'cylindernumber', 'enginesize', 'fuelsystem', 'boreratio', 'stroke',
       'compressionratio', 'horsepower', 'peakrpm', 'citympg', 'highwaympg',
       'price'],
      dtype='object')

- car_ID : 차량 고유 번호 (학습 데이터에서 사용하지 않는 부분)
- symboling : 차량 보험 위험도 지수 ( -3 ~ 3 ) 
    - -3 : 가장 안전함
    - 3 : 가장 위험함
    - 숫자가 높을수록 위험도 크다 (스포츠카가 높은 성향)
- CarName : 자동차 브랜드, 모델명
- fueltype : 연료 종류 (gas(가솔린/휘발유), deisel(디젤))
- aspiration : 엔진 흡기 방식 (std, turbo)
- doornumber : 문 개수
- carbody : 차량 형태 (sedan, hatchback, wagon)
- drivewheel : 구동 방식 (fwd, rwd, 4wd)
- enginelocation : 엔진 위치 (front, rear)
- wheelbase : 휠베이스 ( 앞바퀴와 뒷바퀴 사이의 거리 )
- carlength : 차량 전체 길이
- carwidth : 차량 전체 너비 
- carheight : 차량 높이 
- curbweight : 공차 중량
- enginetype : 엔진 종류 (ohc, ohcv, l, rotor)
- cylindernumber : 실린더 개수 ( four, six, eight )
- enginesize : 배기량
- fuelsystem : 연료 분사 시스템 ( mpfi, 2bbl )
- boreratio : 보어 비율 (실린더 지름의 비율)
- stroke : 스트로크 (피스톤이 상하로 움직이는 거리)
- compressionratio : 압축비 (공기와 연료를 얼마나 압축 비율)
- horsepower : 마력 
- peakrpm : 최대 RPM
- citympg : 도심 연비
- highwaympg : 고속도로 연비
- price : 차량의 가격 ( target )

In [105]:
# 데이터를 문자형 데이터와 숫자형 데이터를 나눠준다. 
# 특정 타입의 컬럼들만 선택한다. select + dtypes
df_str = car.select_dtypes('object')
df_int = car.select_dtypes('number')

In [106]:
df_str.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   CarName         205 non-null    object
 1   fueltype        205 non-null    object
 2   aspiration      205 non-null    object
 3   doornumber      205 non-null    object
 4   carbody         205 non-null    object
 5   drivewheel      205 non-null    object
 6   enginelocation  205 non-null    object
 7   enginetype      205 non-null    object
 8   cylindernumber  205 non-null    object
 9   fuelsystem      205 non-null    object
dtypes: object(10)
memory usage: 16.1+ KB


In [107]:
df_int.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   car_ID            205 non-null    int64  
 1   symboling         205 non-null    int64  
 2   wheelbase         205 non-null    float64
 3   carlength         205 non-null    float64
 4   carwidth          205 non-null    float64
 5   carheight         205 non-null    float64
 6   curbweight        205 non-null    int64  
 7   enginesize        205 non-null    int64  
 8   boreratio         205 non-null    float64
 9   stroke            205 non-null    float64
 10  compressionratio  205 non-null    float64
 11  horsepower        205 non-null    int64  
 12  peakrpm           205 non-null    int64  
 13  citympg           205 non-null    int64  
 14  highwaympg        205 non-null    int64  
 15  price             205 non-null    float64
dtypes: float64(8), int64(8)
memory usage: 25.8 K

In [108]:
df_int.describe()

,car_ID,symboling,wheelbase,carlength,carwidth,carheight,curbweight,enginesize,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
count,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000
mean,103.000000,0.834146,98.756585,174.049268,65.907805,53.724878,2555.565854,126.907317,3.329756,3.255415,10.142537,104.117073,5125.121951,25.219512,30.751220,13276.710571
std,59.322565,1.245307,6.021776,12.337289,2.145204,2.443522,520.680204,41.642693,0.270844,0.313597,3.972040,39.544167,476.985643,6.542142,6.886443,7988.852332
min,1.000000,-2.000000,86.600000,141.100000,60.300000,47.800000,1488.000000,61.000000,2.540000,2.070000,7.000000,48.000000,4150.000000,13.000000,16.000000,5118.000000
25%,52.000000,0.000000,94.500000,166.300000,64.100000,52.000000,2145.000000,97.000000,3.150000,3.110000,8.600000,70.000000,4800.000000,19.000000,25.000000,7788.000000
50%,103.000000,1.000000,97.000000,173.200000,65.500000,54.100000,2414.000000,120.000000,3.310000,3.290000,9.000000,95.000000,5200.000000,24.000000,30.000000,10295.000000
75%,154.000000,2.000000,102.400000,183.100000,66.900000,55.500000,2935.000000,141.000000,3.580000,3.410000,9.400000,116.000000,5500.000000,30.000000,34.000000,16503.000000
max,205.000000,3.000000,120.900000,208.100000,72.300000,59.800000,4066.000000,326.000000,3.940000,4.170000,23.000000,288.000000,6600.000000,49.000000,54.000000,45400.000000


In [109]:
df_str.head()

,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,enginetype,cylindernumber,fuelsystem
0,alfa-romero giulia,gas,std,two,convertible,rwd,front,dohc,four,mpfi
1,alfa-romero stelvio,gas,std,two,convertible,rwd,front,dohc,four,mpfi
2,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,ohcv,six,mpfi
3,audi 100 ls,gas,std,four,sedan,fwd,front,ohc,four,mpfi
4,audi 100ls,gas,std,four,sedan,4wd,front,ohc,five,mpfi


In [110]:
# doornumber 데이터컬럼을 숫자형 데이터로 변환
df_str['doornumber'].unique()

array(['two', 'four'], dtype=object)

In [111]:
df_str['doornumber'] = df_str['doornumber'].map(
    lambda x : 2 if x == 'two' else 4
)

In [112]:
df_str['cylindernumber'].unique()

array(['four', 'six', 'five', 'three', 'twelve', 'two', 'eight'],
      dtype=object)

In [113]:
df_str['cylindernumber'] = df_str['cylindernumber'].map(
    {
        'four' : 4, 
        'six' : 6, 
        'five' : 5, 
        'three' : 3, 
        'twelve' : 12, 
        'two' : 2, 
        'eight' : 8
    }
)

In [ ]:
df_str.info()

In [117]:
# carName 컬럼에서 브랜드 부분에서 제조사와 모델로 새로운 컬럼을 생성
# carName에서 브랜드와 모델명을 나눠준다. ( 빈칸을 기준으로 문자열을 자른다 )
vals = df_str['CarName'].map(
    lambda x :  x.split()
).values

In [ ]:
# df['col'] --> 1차원 Series
# df['new col'] = 1차원 데이터 (list, Series)

# df[ ['col1', 'col2'] ] --> 2차원 DataFrame
# df[ ['new col1', 'new col2'] ] = 2차원 데이터

In [ ]:
df_str[ [ 'brand', 'modelName' ] ] = vals

In [ ]:
# 특정 문자의 위치를 찾는다 ->  find() / index()

In [124]:
vals2 = list(
    map(
        # lambda x : x.split()[:2] ,
        lambda x : [ x[ : x.find(' ')] ,  x[x.find(' ') :].strip() ] ,
        df_str['CarName']
    )
)

In [125]:
df_str[ [ 'brand', 'modelName' ] ] = vals2

In [ ]:
df_str.head()

In [128]:
# CarName과 modelName은 제거 
df_str.drop(['CarName', 'modelName'], axis=1, inplace=True)

In [129]:
df_str['brand'].unique()

array(['alfa-romero', 'audi', 'bmw', 'chevrolet', 'dodge', 'honda',
       'isuzu', 'jaguar', 'maxda', 'mazda', 'buick', 'mercury',
       'mitsubishi', 'Nissan', 'nissan', 'peugeot', 'plymouth', 'porsche',
       'porcshce', 'renault', 'saab', 'subar', 'subaru', 'toyota',
       'toyouta', 'vokswagen', 'volkswagen', 'vw', 'volvo'], dtype=object)

In [130]:
# df_str에서 object형 컬럼들을 모두 더비 변수 생성 
cols = df_str.select_dtypes('object').columns

df_str2 = pd.get_dummies(df_str, columns=cols, drop_first=True)

In [131]:
df_str2

,doornumber,cylindernumber,fueltype_gas,aspiration_turbo,carbody_hardtop,carbody_hatchback,carbody_sedan,carbody_wagon,drivewheel_fwd,drivewheel_rwd,...,brand_renault,brand_saab,brand_subar,brand_subaru,brand_toyota,brand_toyouta,brand_vokswagen,brand_volkswagen,brand_volvo,brand_vw
0,2,4,True,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
1,2,4,True,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
2,2,6,True,False,False,True,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
3,4,4,True,False,False,False,True,False,True,False,...,False,False,False,False,False,False,False,False,False,False
4,4,5,True,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,4,4,True,False,False,False,True,False,False,True,...,False,False,False,False,False,False,False,False,True,False
201,4,4,True,True,False,False,True,False,False,True,...,False,False,False,False,False,False,False,False,True,False
202,4,6,True,False,False,False,True,False,False,True,...,False,False,False,False,False,False,False,False,True,False
203,4,6,False,True,False,False,True,False,False,True,...,False,False,False,False,False,False,False,False,True,False


In [135]:
df_int.drop('car_ID', axis=1, inplace=True)

In [137]:
# df_str2와 df_int를 단순 열 결합 
df = pd.concat( [df_str2, df_int], axis=1 )

In [138]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 67 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   doornumber           205 non-null    int64  
 1   cylindernumber       205 non-null    int64  
 2   fueltype_gas         205 non-null    bool   
 3   aspiration_turbo     205 non-null    bool   
 4   carbody_hardtop      205 non-null    bool   
 5   carbody_hatchback    205 non-null    bool   
 6   carbody_sedan        205 non-null    bool   
 7   carbody_wagon        205 non-null    bool   
 8   drivewheel_fwd       205 non-null    bool   
 9   drivewheel_rwd       205 non-null    bool   
 10  enginelocation_rear  205 non-null    bool   
 11  enginetype_dohcv     205 non-null    bool   
 12  enginetype_l         205 non-null    bool   
 13  enginetype_ohc       205 non-null    bool   
 14  enginetype_ohcf      205 non-null    bool   
 15  enginetype_ohcv      205 non-null    boo

In [139]:
x = df.drop('price', axis=1)
y = df['price']

In [140]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, random_state=42
)

In [141]:
base_model = DecisionTreeRegressor()
reg = BaggingRegressor(estimator=base_model, n_estimators=100, oob_score=True)

In [142]:
reg.fit(X_train, y_train)

,"estimator estimator: object, default=NoneThe base estimator to fit on random subsets of the dataset.If None, then the base estimator is a:class:`~sklearn.tree.DecisionTreeRegressor`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",DecisionTreeRegressor()
,"n_estimators n_estimators: int, default=10The number of base estimators in the ensemble.",100
,"max_samples max_samples: int or float, default=NoneThe number of samples to draw from X to train each base estimator (withreplacement by default, see `bootstrap` for more details).- If None, then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples.",None
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator (without replacement by default, see `bootstrap_features` for moredetails).- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.",1.0
,"bootstrap bootstrap: bool, default=TrueWhether samples are drawn with replacement. If False, sampling withoutreplacement is performed. If fitting with `sample_weight`, it isstrongly recommended to choose True, as only drawing with replacementwill ensure the expected frequency semantics of `sample_weight`.",True
,"bootstrap_features bootstrap_features: bool, default=FalseWhether features are drawn with replacement.",False
,"oob_score oob_score: bool, default=FalseWhether to use out-of-bag samples to estimatethe generalization error. Only available if bootstrap=True.",True
,"warm_start warm_start: bool, default=FalseWhen set to True, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fita whole new ensemble. See :term:`the Glossary `.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for both :meth:`fit` and:meth:`predict`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random resampling of the original dataset(sample wise and feature wise).If the base estimator accepts a `random_state` attribute, a differentseed is generated for each instance in the ensemble.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",0


In [143]:
reg.oob_score_

0.9035456210018622

In [150]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [145]:
pred = reg.predict(X_test)

In [151]:
mse = mean_squared_error(y_test, pred)
r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)

In [152]:
print('MAE : ', round(mae, 2))
print("MSE : ", round(mse, 2))
print('R2 : ', round(r2, 2))


MAE :  1305.38
MSE :  3774372.16
R2 :  0.95


In [149]:
df['price'].describe()

count      205.000000
mean     13276.710571
std       7988.852332
min       5118.000000
25%       7788.000000
50%      10295.000000
75%      16503.000000
max      45400.000000
Name: price, dtype: float64